# 23 — Threshold Selection & Operational Metrics

Loads the trained CatBoost model and predictions saved by NB22.

1. **Threshold selection** — sweep percentile cutoffs on val (2021), pick max F2, report test performance  
2. **Operational metrics** — alert rate, lead time per depeg event, false-alert days


In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Capstone')
else:
    ROOT = Path.cwd()
    if not (ROOT / 'data').exists():
        ROOT = ROOT.parent

MODELS_DIR   = ROOT / 'data' / 'models'
FEATURES_DIR = ROOT / 'data' / 'processed' / 'features'
print(f'ROOT: {ROOT}')


ROOT: /Users/robertspringett/Education/CMU_MSBA/capstone_5min


## 1. Load Model & Predictions


In [2]:
# Load metadata
with open(MODELS_DIR / 'nb10_meta.json') as f:
    meta = json.load(f)

TARGET      = meta['target']
best_params = meta['best_params']
feat_full   = meta['feat_full']
base        = meta['base_rate']
EMBARGO_END = pd.Timestamp(meta['embargo_end'])
VAL_START   = pd.Timestamp(meta['val_start'])

# Load predictions
pred_te  = pd.read_parquet(MODELS_DIR / 'nb10_test_predictions.parquet')
pred_val = pd.read_parquet(MODELS_DIR / 'nb10_val_predictions.parquet')
pred_te.index  = pd.to_datetime(pred_te.index,  utc=True)
pred_val.index = pd.to_datetime(pred_val.index, utc=True)

p_all      = pred_te['proba'].values
y_te       = pred_te['y_true'].values
p_val      = pred_val['proba'].values
y_val_tune = pred_val['y_true'].values
base_val   = float(y_val_tune.mean())

# Load full df for onset detection
df = pd.read_parquet(FEATURES_DIR / 'pooled_5m.parquet')
df.index = pd.to_datetime(df.index, utc=True)
te = pred_te  # alias — has coin + index

n_days_te    = len(pred_te.index.normalize().unique())
n_days_val   = len(pred_val.index.normalize().unique())

print(f'Test  : {len(pred_te):,} bars  pos={int(y_te.sum())}  base={base*100:.3f}%')
print(f'Val   : {len(pred_val):,} bars  pos={int(y_val_tune.sum())}  base={base_val*100:.3f}%')
print(f'Model : CatBoost  iterations={best_params["iterations"]}  depth={best_params["depth"]}')


Test  : 1,730,601 bars  pos=736  base=0.043%
Val   : 518,568 bars  pos=836  base=0.161%
Model : CatBoost  iterations=200  depth=4


## 2. Threshold Selection

Optimal threshold found by maximising F2 on the **validation set (2021)** — never touches the test set.
Test performance is then reported at that fixed threshold.


In [3]:
print(f'Validation (2021): {len(y_val_tune):,} bars  pos={int(y_val_tune.sum())}  base={base_val*100:.3f}%')
print()

best_f2_val, best_pct = 0, None
print(f'{"Pct cut":>8}  {"Alerts/day":>11}  {"Recall":>8}  {"Precision":>10}  {"Lift":>7}  {"F2":>8}  (VAL)')
print('-' * 68)
for pct in [0.1, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0]:
    t = np.percentile(p_val, 100 - pct)
    al = p_val >= t
    if al.sum() == 0: continue
    tp   = int((al & (y_val_tune == 1)).sum())
    prec = tp / al.sum()
    rec  = tp / int(y_val_tune.sum())
    f2   = (5*prec*rec)/(4*prec+rec) if (prec+rec) > 0 else 0
    apd  = al.sum() / n_days_val
    marker = ' <- best' if f2 > best_f2_val else ''
    if f2 > best_f2_val:
        best_f2_val, best_pct = f2, pct
    print(f'top {pct:>4.1f}%  {apd:>11.1f}  {rec*100:>7.1f}%  {prec*100:>9.2f}%  {prec/base_val:>6.1f}x  {f2:>8.4f}{marker}')

print()
print(f'Val-optimal percentile: top {best_pct}%  (F2={best_f2_val:.4f})')
print()

# Apply to test set
t_te  = np.percentile(p_all, 100 - best_pct)
al_te = p_all >= t_te
apd_te   = al_te.sum() / n_days_te
tp_te    = int((al_te & (y_te == 1)).sum())
prec_te  = tp_te / al_te.sum() if al_te.sum() > 0 else 0
rec_te   = tp_te / int(y_te.sum())
f2_te    = (5*prec_te*rec_te)/(4*prec_te+rec_te) if (prec_te+rec_te) > 0 else 0

print(f'Test performance at val-optimal threshold (top {best_pct}%):')
print(f'  Alerts/day : {apd_te:.1f}')
print(f'  Recall     : {rec_te*100:.1f}%')
print(f'  Precision  : {prec_te*100:.2f}%')
print(f'  Lift       : {prec_te/base:.1f}x')
print(f'  F2 score   : {f2_te:.4f}')
print()

# Optimism check
best_f2_te = 0
for pct in [0.1, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0]:
    t = np.percentile(p_all, 100 - pct)
    al = p_all >= t
    if al.sum() == 0: continue
    tp   = int((al & (y_te == 1)).sum())
    prec = tp / al.sum()
    rec  = tp / int(y_te.sum())
    f2   = (5*prec*rec)/(4*prec+rec) if (prec+rec) > 0 else 0
    if f2 > best_f2_te: best_f2_te = f2

print(f'Oracle F2 (best percentile on test): {best_f2_te:.4f}')
print(f'Val-selected F2 on test:             {f2_te:.4f}')
print(f'Optimism gap:                        {best_f2_te - f2_te:.4f}')


Validation (2021): 518,568 bars  pos=836  base=0.161%

 Pct cut   Alerts/day    Recall   Precision     Lift        F2  (VAL)
--------------------------------------------------------------------
top  0.1%          1.4     27.8%      44.44%   275.7x    0.3001 <- best
top  0.2%          3.6     50.7%      32.69%   202.8x    0.4568 <- best
top  0.5%          7.1     76.0%      24.49%   151.9x    0.5348 <- best
top  1.0%         14.2     95.1%      15.33%    95.1x    0.4660
top  2.0%         28.4     99.8%       8.04%    49.9x    0.3040
top  3.0%         42.6    100.0%       5.37%    33.3x    0.2211
top  5.0%         71.0    100.0%       3.22%    20.0x    0.1428

Val-optimal percentile: top 0.5%  (F2=0.5348)

Test performance at val-optimal threshold (top 0.5%):
  Alerts/day : 5.8
  Recall     : 71.5%
  Precision  : 6.08%
  Lift       : 142.9x
  F2 score   : 0.2267



Oracle F2 (best percentile on test): 0.4438
Val-selected F2 on test:             0.2267
Optimism gap:                        0.2170


## 3. Operational Metrics

Three deployment-facing metrics at the val-optimal threshold:

1. **Alert rate** — fraction of test bars that trigger an alert  
2. **Lead time per event** — how early the model fires before each depeg onset (per coin)  
3. **False-alert days** — days with >=1 alert but no true depeg within 4 h of any alert bar


In [4]:
alerts_s = pd.Series(al_te.astype(bool), index=pred_te.index)

print(f'Threshold: top {best_pct}% (val-optimal)')
print('=' * 62)

# ── 1. Alert Rate ─────────────────────────────────────────────────────────────
alert_rate = al_te.sum() / len(pred_te)
print(f'\n1. ALERT RATE')
print(f'   Alerted bars  : {int(al_te.sum()):,} / {len(pred_te):,}  ({alert_rate*100:.3f}%)')
print(f'   Avg alerts/day: {al_te.sum() / n_days_te:.1f}')

# ── 2. Lead Time per Depeg Event ──────────────────────────────────────────────
te_full  = df[df.index >= EMBARGO_END]
LOOKBACK = pd.Timedelta('24h')

alerts_by_coin = {}
for _coin in pred_te['coin'].unique():
    _mask = pred_te['coin'] == _coin
    alerts_by_coin[_coin] = alerts_s[_mask]

onset_rows = []
for coin, grp in te_full.groupby('coin'):
    dep = grp.sort_index()['depeg'].fillna(0).astype(int)
    onset_mask = (dep == 1) & (dep.shift(1).fillna(0).astype(int) == 0)
    for t in dep.index[onset_mask]:
        onset_rows.append({'coin': coin, 'onset': t})

onset_df = pd.DataFrame(onset_rows).sort_values('onset').reset_index(drop=True)
print(f'\n2. LEAD TIME PER DEPEG EVENT  ({len(onset_df)} onsets in test period)')
print(f'   {"Onset (UTC)":<26}  {"Coin":>6}  {"Lead time":>22}  First alert (UTC)')
print(f'   {"-"*26}  {"-"*6}  {"-"*22}  {"-"*24}')

lead_times = []
for _, row in onset_df.iterrows():
    coin, onset = row['coin'], row['onset']
    coin_a  = alerts_by_coin.get(coin, pd.Series(dtype=bool))
    window  = coin_a[(coin_a.index >= onset - LOOKBACK) & (coin_a.index < onset)]
    fired   = window[window]
    if len(fired) > 0:
        first_alert = fired.index[0]
        lead_m = (onset - first_alert).total_seconds() / 60
        lead_str = f'{lead_m:.0f} min ({lead_m/60:.1f} h)'
        lead_times.append(lead_m)
    else:
        first_alert = None
        lead_str = 'no alert'
    print(f'   {str(onset)[:25]}  {coin:>6}  {lead_str:>22}  {str(first_alert)[:24] if first_alert else "---"}')

if lead_times:
    print(f'\n   Events alerted  : {len(lead_times)} / {len(onset_df)}  ({100*len(lead_times)/len(onset_df):.0f}%)')
    print(f'   Median lead time: {np.median(lead_times):.0f} min  ({np.median(lead_times)/60:.1f} h)')
    print(f'   Min / Max       : {min(lead_times):.0f} / {max(lead_times):.0f} min')

# ── 3. False-Alert Days ───────────────────────────────────────────────────────
print(f'\n3. FALSE-ALERT DAYS')
date_idx = pred_te.index.normalize()
day_df = pd.DataFrame({
    'alert':    al_te.astype(int),
    'true_pos': (al_te & y_te.astype(bool)).astype(int),
}, index=date_idx)
day_summary      = day_df.groupby(level=0).max()
days_alerted     = int(day_summary['alert'].sum())
days_true_alert  = int(day_summary['true_pos'].sum())
days_false_only  = days_alerted - days_true_alert

print(f'   Total test days        : {n_days_te}')
print(f'   Days with any alert    : {days_alerted}  ({100*days_alerted/n_days_te:.1f}%)')
print(f'   Days with true alert   : {days_true_alert}  ({100*days_true_alert/n_days_te:.1f}%)')
print(f'   False-alert-only days  : {days_false_only}  ({100*days_false_only/n_days_te:.1f}%)')
print(f'   Silent days (no alert) : {n_days_te - days_alerted}  ({100*(n_days_te-days_alerted)/n_days_te:.1f}%)')


Threshold: top 0.5% (val-optimal)

1. ALERT RATE
   Alerted bars  : 8,655 / 1,730,601  (0.500%)
   Avg alerts/day: 5.8



2. LEAD TIME PER DEPEG EVENT  (101 onsets in test period)
   Onset (UTC)                   Coin               Lead time  First alert (UTC)
   --------------------------  ------  ----------------------  ------------------------
   2022-02-22 12:50:00+00:00     ust                no alert  ---
   2022-02-22 18:10:00+00:00     ust                no alert  ---
   2022-02-22 20:25:00+00:00     ust                no alert  ---
   2022-02-22 21:10:00+00:00     ust                no alert  ---
   2022-02-23 00:15:00+00:00     ust                no alert  ---
   2022-02-25 22:25:00+00:00     ust                no alert  ---
   2022-02-25 23:10:00+00:00     ust                no alert  ---
   2022-02-25 23:30:00+00:00     ust                no alert  ---
   2022-03-02 21:50:00+00:00     ust                no alert  ---
   2022-03-02 23:20:00+00:00     ust                no alert  ---
   2022-03-04 20:40:00+00:00     ust                no alert  ---
   2022-03-04 21:05:00+00:00     ust          

## 4. Operational Metrics — Market-Driven Events Only

Excludes UST (Terra collapse, May 2022) and BUSD (Paxos halt, Feb 2023) — both were
regulatory/structural failures, not open-market stress. This gives a cleaner read on
how the model performs on genuinely market-driven depegs.


In [5]:
EXCLUDE_COINS = {'ust', 'busd'}

# Filter predictions to market-driven coins only
mask_mkt   = ~pred_te['coin'].isin(EXCLUDE_COINS)
pred_mkt   = pred_te[mask_mkt]
al_mkt     = al_te[mask_mkt.values]
y_mkt      = y_te[mask_mkt.values]
alerts_mkt = pd.Series(al_mkt.astype(bool), index=pred_mkt.index)
n_days_mkt = len(pred_mkt.index.normalize().unique())
base_mkt   = float(y_mkt.mean())

print(f'Market-driven coins: {sorted(set(pred_te["coin"].unique()) - EXCLUDE_COINS)}')
print(f'Test bars (excl. UST/BUSD): {len(pred_mkt):,}  pos={int(y_mkt.sum())}  base={base_mkt*100:.3f}%')
print(f'Threshold: top {best_pct}% (val-optimal)')
print('=' * 62)

# ── 1. Alert Rate ─────────────────────────────────────────────────────────────
print(f'\n1. ALERT RATE')
print(f'   Alerted bars  : {int(al_mkt.sum()):,} / {len(pred_mkt):,}  ({100*al_mkt.sum()/len(pred_mkt):.3f}%)')
print(f'   Avg alerts/day: {al_mkt.sum() / n_days_mkt:.1f}')

# ── 2. Lead Time per Depeg Event ──────────────────────────────────────────────
alerts_by_coin_mkt = {}
for _coin in pred_mkt['coin'].unique():
    _mask = pred_mkt['coin'] == _coin
    alerts_by_coin_mkt[_coin] = alerts_mkt[_mask]

onset_mkt = onset_df[~onset_df['coin'].isin(EXCLUDE_COINS)].reset_index(drop=True)

print(f'\n2. LEAD TIME PER DEPEG EVENT  ({len(onset_mkt)} onsets, excl. UST/BUSD)')
print(f'   {"Onset (UTC)":<26}  {"Coin":>6}  {"Lead time":>22}  First alert (UTC)')
print(f'   {"-"*26}  {"-"*6}  {"-"*22}  {"-"*24}')

lead_times_mkt = []
for _, row in onset_mkt.iterrows():
    coin, onset = row['coin'], row['onset']
    coin_a  = alerts_by_coin_mkt.get(coin, pd.Series(dtype=bool))
    window  = coin_a[(coin_a.index >= onset - LOOKBACK) & (coin_a.index < onset)]
    fired   = window[window]
    if len(fired) > 0:
        first_alert = fired.index[0]
        lead_m = (onset - first_alert).total_seconds() / 60
        lead_str = f'{lead_m:.0f} min ({lead_m/60:.1f} h)'
        lead_times_mkt.append(lead_m)
    else:
        first_alert = None
        lead_str = 'no alert'
    print(f'   {str(onset)[:25]}  {coin:>6}  {lead_str:>22}  {str(first_alert)[:24] if first_alert else "---"}')

if lead_times_mkt:
    print(f'\n   Events alerted  : {len(lead_times_mkt)} / {len(onset_mkt)}  ({100*len(lead_times_mkt)/len(onset_mkt):.0f}%)')
    print(f'   Median lead time: {np.median(lead_times_mkt):.0f} min  ({np.median(lead_times_mkt)/60:.1f} h)')
    print(f'   Min / Max       : {min(lead_times_mkt):.0f} / {max(lead_times_mkt):.0f} min')

# ── 3. False-Alert Days ───────────────────────────────────────────────────────
print(f'\n3. FALSE-ALERT DAYS')
date_idx_mkt = pred_mkt.index.normalize()
day_df_mkt = pd.DataFrame({
    'alert':    al_mkt.astype(int),
    'true_pos': (al_mkt & y_mkt.astype(bool)).astype(int),
}, index=date_idx_mkt)
day_sum_mkt      = day_df_mkt.groupby(level=0).max()
days_alerted_mkt = int(day_sum_mkt['alert'].sum())
days_true_mkt    = int(day_sum_mkt['true_pos'].sum())
days_false_mkt   = days_alerted_mkt - days_true_mkt

print(f'   Total test days        : {n_days_mkt}')
print(f'   Days with any alert    : {days_alerted_mkt}  ({100*days_alerted_mkt/n_days_mkt:.1f}%)')
print(f'   Days with true alert   : {days_true_mkt}  ({100*days_true_mkt/n_days_mkt:.1f}%)')
print(f'   False-alert-only days  : {days_false_mkt}  ({100*days_false_mkt/n_days_mkt:.1f}%)')
print(f'   Silent days (no alert) : {n_days_mkt - days_alerted_mkt}  ({100*(n_days_mkt-days_alerted_mkt)/n_days_mkt:.1f}%)')


Market-driven coins: ['dai', 'rlusd', 'usdc', 'usde', 'usdt']
Test bars (excl. UST/BUSD): 1,580,882  pos=557  base=0.035%
Threshold: top 0.5% (val-optimal)

1. ALERT RATE
   Alerted bars  : 8,237 / 1,580,882  (0.521%)
   Avg alerts/day: 5.5



2. LEAD TIME PER DEPEG EVENT  (52 onsets, excl. UST/BUSD)
   Onset (UTC)                   Coin               Lead time  First alert (UTC)
   --------------------------  ------  ----------------------  ------------------------
   2022-05-12 03:40:00+00:00    usdt        910 min (15.2 h)  2022-05-11 12:30:00+00:0
   2022-05-12 04:30:00+00:00    usdt        960 min (16.0 h)  2022-05-11 12:30:00+00:0
   2022-05-12 05:00:00+00:00    usdt        990 min (16.5 h)  2022-05-11 12:30:00+00:0
   2022-05-12 05:30:00+00:00    usdt       1020 min (17.0 h)  2022-05-11 12:30:00+00:0
   2022-05-12 05:35:00+00:00    usdc       1015 min (16.9 h)  2022-05-11 12:40:00+00:0
   2022-05-12 07:15:00+00:00     dai       1060 min (17.7 h)  2022-05-11 13:35:00+00:0
   2022-11-10 11:10:00+00:00    usdt       1425 min (23.8 h)  2022-11-09 11:25:00+00:0
   2023-03-11 00:55:00+00:00    usdt                no alert  ---
   2023-03-11 01:00:00+00:00    usdc          95 min (1.6 h)  2023-03-10 23:25:00+00:0
   2023-03